In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score
import joblib

# 1. Data Cleaning & Optimization
# Pehli row (header) skip kar rahe hain aur zaroori columns le rahe hain
file_path = '/content/drive/My Drive/archive/training.1600000.processed.noemoticon.csv'
df = pd.read_csv(file_path, encoding='latin-1', header=None, skiprows=1)
df.columns = ['target', 'ids', 'date', 'flag', 'user', 'text']
df = df[['target', 'text']]

# Target ko clean karna (sirf 0 aur 4 rakhna aur convert karna)
df = df[df['target'].isin([0, 4, '0', '4'])] # Sirf valid labels
df['target'] = df['target'].astype(int).replace(4, 1) # 4 ko 1 (Positive) bana diya

# 2. Sampling (Pure 1 million train karne mein ghanto lagenge, 50k is enough for best accuracy)
df_pos = df[df['target'] == 1].sample(25000)
df_neg = df[df['target'] == 0].sample(25000)
df_final = pd.concat([df_pos, df_neg]).sample(frac=1).reset_index(drop=True)

print(f"Final Data for Training: {len(df_final)} rows")

# 3. Vectorization (TF-IDF)
tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=500000)
X = tfidf.fit_transform(df_final['text'])
y = df_final['target']

# 4. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Best Model: LinearSVC (Support Vector Classifier)
model = LinearSVC(C=0.5)
model.fit(X_train, y_train)

# 6. Evaluation
predictions = model.predict(X_test)
print("\n--- Model Performance ---")
print(f"Accuracy: {accuracy_score(y_test, predictions)*100:.2f}%")
print(classification_report(y_test, predictions))

# 7. Model Save karein (Hugging Face ke liye)
joblib.dump(model, 'sentiment_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
print("\n✅ Model and Vectorizer saved successfully!")

Final Data for Training: 50000 rows

--- Model Performance ---
Accuracy: 78.83%
              precision    recall  f1-score   support

           0       0.79      0.78      0.78      4960
           1       0.78      0.80      0.79      5040

    accuracy                           0.79     10000
   macro avg       0.79      0.79      0.79     10000
weighted avg       0.79      0.79      0.79     10000


✅ Model and Vectorizer saved successfully!


In [4]:
# 1. Setup
!pip install -q transformers[torch] datasets evaluate accelerate
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import evaluate

# 2. Data Loading
file_path = '/content/drive/My Drive/archive/training.1600000.processed.noemoticon.csv'
df = pd.read_csv(file_path, encoding='latin-1', header=None)
df.columns = ['labels', 'ids', 'date', 'flag', 'user', 'tweet'] # 'target' ko 'labels' kar diya
df = df[['labels', 'tweet']].copy()
df['labels'] = df['labels'].replace(4, 1)

# Sampling
df_final = pd.concat([
    df[df['labels']==0].sample(10000, random_state=42),
    df[df['labels']==1].sample(10000, random_state=42)
]).sample(frac=1).reset_index(drop=True)

# 3. Tokenization (Important Fix here)
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize_function(examples):
    # Tokenize text and keep labels
    return tokenizer(examples['tweet'], padding="max_length", truncation=True)

# Dataset conversion
train_df, test_df = train_test_split(df_final, test_size=0.15, random_state=42)
train_dataset = Dataset.from_dict(train_df)
test_dataset = Dataset.from_dict(test_df)

# Map labels correctly
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# 4. Model & Metrics
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 5. Training Arguments
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

# 6. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("\n🚀 Starting Training... Ab Error nahi aayega!")
trainer.train()

# Final Result
final_eval = trainer.evaluate()
print(f"\n✅ Final BERT Accuracy: {final_eval['eval_accuracy']*100:.2f}%")


/tmp/ipykernel_1057/2556958770.py:13: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding='latin-1', header=None)


Map:   0%|          | 0/17000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



✅ Final BERT Accuracy: 82.67%


In [4]:
%%writefile app.py
import streamlit as st
import pickle
import re

# Load model
model = pickle.load(open('sentiment_model.pkl', 'rb'))
vectorizer = pickle.load(open('tfidf_vectorizer.pkl', 'rb'))

# Clean text
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

# Predict
def predict_sentiment(text):
    text = clean_text(text)
    vector = vectorizer.transform([text])
    prediction = model.predict(vector)
    return prediction[0]

# Page config
st.set_page_config(page_title="AI Sentiment Analyzer", page_icon="🤖", layout="centered")

# Custom CSS (UI styling 🔥)
st.markdown("""
    <style>
    .main {
        background-color: #f5f7fa;
    }
    .title {
        text-align: center;
        font-size: 40px;
        font-weight: bold;
        color: #4A90E2;
    }
    .subtitle {
        text-align: center;
        font-size: 18px;
        color: gray;
        margin-bottom: 30px;
    }
    .result-box {
        padding: 15px;
        border-radius: 10px;
        text-align: center;
        font-size: 20px;
        font-weight: bold;
    }
    </style>
""", unsafe_allow_html=True)

# Title
st.markdown('<div class="title">🤖 AI Sentiment Analyzer</div>', unsafe_allow_html=True)
st.markdown('<div class="subtitle">Analyze text sentiment instantly</div>', unsafe_allow_html=True)

# Input box
user_input = st.text_area("✍️ Enter your text here:", height=150)

# Button centered
col1, col2, col3 = st.columns([1,2,1])
with col2:
    analyze = st.button("🔍 Analyze Sentiment")

# Prediction
if analyze:
    if user_input.strip() == "":
        st.warning("⚠️ Please enter some text")
    else:
        result = predict_sentiment(user_input)

        if result == 1:
            st.markdown(
                '<div class="result-box" style="background-color:#d4edda; color:#155724;">😊 Positive Sentiment</div>',
                unsafe_allow_html=True
            )
        else:
            st.markdown(
                '<div class="result-box" style="background-color:#f8d7da; color:#721c24;">😡 Negative Sentiment</div>',
                unsafe_allow_html=True
            )


Writing app.py
